# L3: Multi-agent Customer Support Automation

In this lesson, you will learn about the six key elements which help make Agents perform even better:
- Role Playing
- Focus
- Tools
- Cooperation
- Guardrails
- Memory

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
# 直接传 Memory 给 Crew 时，需关闭遥测（否则会报 Invalid type Memory）
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"

from crewai import Agent, Task, Crew, LLM

- Import libraries, API and LLM

In [3]:
from crewai import Agent, Task, Crew

In [4]:
from utils import load_env
load_env()

## Role Playing, Focus and Cooperation

In [5]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="成为团队中最友好、最有帮助的支持代表",
	backstory=(
		"你在 crewAI (https://crewai.com) 工作，"
        "现在正在为 {customer} 提供支持，"
		"这是公司非常重要的客户。"
		"你需要确保提供最好的支持！"
		"务必提供完整、全面的答案，不要做任何假设。"
	),
	allow_delegation=False,
	verbose=True
)

- By not setting `allow_delegation=False`, `allow_delegation` takes its default value of being `True`.
- This means the agent _can_ delegate its work to another agent which is better suited to do a particular task. 

In [6]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="因提供团队中最优质的支持质量保证而获得认可",
	backstory=(
		"你在 crewAI (https://crewai.com) 工作，"
        "现在正与团队合作处理来自 {customer} 的请求，"
		"确保支持代表提供尽可能最好的支持。\n"
		"你需要确保支持代表提供完整、全面的答案，"
        "不要做任何假设。"
	),
	verbose=True
)

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

## Tools, Guardrails and Memory

### Tools

- Import CrewAI tools

In [7]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

### Possible Custom Tools
- Load customer data
- Tap into previous conversations
- Load data from a CRM
- Checking existing bug reports
- Checking existing feature requests
- Checking ongoing tickets
- ... and more

- Some ways of using CrewAI tools.

```Python
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
```

- Instantiate a document scraper tool.
- The tool will scrape a page (only 1 URL) of the CrewAI documentation.

In [8]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/en/enterprise/guides/kickoff-crew"
)

##### Different Ways to Give Agents Tools

- Agent Level: The Agent can use the Tool(s) on any Task it performs.
- Task Level: The Agent will only use the Tool(s) when performing that specific Task.

**Note**: Task Tools override the Agent Tools.

### Creating Tasks
- You are passing the Tool on the Task Level.

In [9]:
inquiry_resolution = Task(
    description=(
        "{customer} 刚刚提出了一个非常重要的请求：\n"
	    "{inquiry}\n\n"
        "联系人是 {customer} 的 {person}。"
		"请运用你所掌握的一切知识，"
        "尽可能提供最好的支持。"
		"你必须努力对客户的问题给出完整、准确的回复。"
    ),
    expected_output=(
	    "对客户询问的详细、有信息量的回复，"
        "涵盖其问题的各个方面。\n"
        "回复应包含用于查找答案的所有参考资料，"
        "包括外部数据或解决方案。"
        "确保答案完整、不留未解答的问题，"
		"并在整个过程中保持友善、友好的语气。"
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent

In [10]:
quality_assurance_review = Task(
    description=(
        "审阅高级支持代表为 {customer} 的询问起草的回复。"
        "确保答案全面、准确，并符合客户支持的高质量标准。\n"
        "验证客户询问的所有部分是否已得到充分回应，"
        "语气友好、亲切。\n"
        "检查用于查找信息的参考资料和来源，"
		"确保回复有充分依据，且不留未解答的问题。"
    ),
    expected_output=(
        "一份可直接发送给客户的最终、详细且有信息量的回复。\n"
        "该回复应充分回应客户的询问，"
        "并纳入所有相关反馈与改进。\n"
		"不必过于正式——我们是一家轻松、时髦的公司，"
	    "但全程仍应保持专业、友好的语气。"
    ),
    agent=support_quality_assurance_agent,
)


### Creating the Crew

#### Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [11]:
import os
from crewai.memory.unified_memory import Memory

llm = LLM(model=os.environ.get("OPENAI_MODEL_NAME", "qwen-turbo"))
print("model:", os.environ.get("OPENAI_MODEL_NAME"))


memory_embedder = {
    "provider": "openai",
    "config": {
        "model_name": os.environ.get("EMBEDDINGS_OPENAI_MODEL_NAME", "text-embedding-v3"),
        "api_base": os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE"),
        "api_key": os.environ.get("OPENAI_API_KEY"),
    },
}

crew_memory = Memory(
    llm=llm,
    embedder=memory_embedder,
)

crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=True,
  embedder=memory_embedder,
)


model: qwen3.5-27b


### Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

#### Guardrails
- By running the execution below, you can see that the agents and the responses are within the scope of what we expect from them.

In [12]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "我需要帮助设置 Crew 并启动它，"
               "具体来说，如何为我的 crew 添加记忆？"
               "你能提供指导吗？"
}
result = await crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fce04721-ad36-4aa6-b81a-b470d289f3e8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: DeepLearningAI 刚刚提出了一个非常重要的请求：                                                            │
│  我需要帮助设置 Crew 并启动它，具体来说，如何为我的 crew 添加记忆？你能提供指导吗？                             │
│                                                                                                                 │
│  联系人是 DeepLearningAI 的 Andrew                                                                              │
│  Ng。请运用你所掌握的一切知识，尽可能提供最好的支持。你必须努力对客户的问题给出完整、准确的回复。               │
│  ID: 30fb3e86-1343-437b-98c6-53359d804e75                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: DeepLearningAI 刚刚提出了一个非常重要的请求：                                                            │
│  我需要帮助设置 Crew 并启动它，具体来说，如何为我的 crew 添加记忆？你能提供指导吗？                             │
│                                                                                                                 │
│  联系人是 DeepLearningAI 的 Andrew                                                                              │
│  Ng。请运用你所掌握的一切知识，尽可能提供最好的支持。你必须努力对客户的问题给出完整、准确的回复。               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['crew memory', 'CrewAI memory setup', 'adding memory to crew']}                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_website_content                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Model gpt-4o-mini not found: Error code: 404 - {'error': {'message': 'The model `gpt-4o-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'd251fd5c-157e-9b47-ba24-0e1bf5b20fdf'}
ERROR:root:OpenAI API call failed: Model gpt-4o-mini not found: Error code: 404 - {'error': {'message': 'The model `gpt-4o-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': 'd251fd5c-157e-9b47-ba24-0e1bf5b20fdf'}


[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'memory_save_started' (expected 
'llm_call_started')

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: Model gpt-4o-mini not found: Error code: 404 - {'error': {'message': 'The model `gpt-4o-mini` does not  │
│  exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code':               │
│  'model_not_found'}, 'request_id': 'd251fd5c-157e-9b47-ba24-0e1bf5b20fdf'}                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Model gpt-4o-mini not found: Error code: 404 - {'error': {'message': 'The       │
│  model `gpt-4o-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error',         │
│  'param': None, 'code': 'model_not_found'}, 'request_id': 'd251fd5c-157e-9b47-ba24-0e1bf5b20fdf'}               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [13]:
from IPython.display import Markdown
Markdown(result.raw)

# Dear Andrew Ng,

感谢您联系 CrewAI 支持团队！我是您的专属支持代表，很高兴能为您提供帮助。您提出的关于为 Crew 添加记忆的问题非常重要，我很乐意为您提供详细、全面的指导。

## 📋 完整的 CrewAI 记忆配置指南

### 🔍 什么是 CrewAI 记忆？

在 CrewAI 中，**记忆（Memory）**功能让 AI Agents 能够记住先前的交互、任务结果和上下文信息，从而在长时间运行的会话中保持一致性和智能性。记忆系统包含三种主要类型：

1. **Short-Term Memory（短期记忆）** - 存储当前会话中的即时信息
2. **Long-Term Memory（长期记忆）** - 持久化重要的信息和模式
3. **Entity Memory（实体记忆）** - 跟踪特定实体的相关信息

---

## 🛠️ 如何为您的 Crew 添加记忆 - 完整步骤

### 第 1 步：安装必要的依赖

首先确保您的环境中安装了所有需要的包：

```bash
pip install crewai==0.14.4 # 或最新版本
pip install chromadb       # 用于长期记忆的向量数据库
```

> 💡 **提示**：如果您使用的是 Docker 环境，请确保 ChromaDB 容器也在运行。

### 第 2 步：配置记忆系统

#### 方法 A：使用默认配置（推荐初学者）

```python
from crewai import Crew, Agent, Task
from crewai.memory import LongTermMemory, ShortTermMemory

# 创建具有记忆功能的 Agent
research_agent = Agent(
    role='资深研究分析师',
    goal='深入研究技术趋势并识别关键见解',
    backstory="""作为科技行业领先的分析师，你专注于人工智能和深度学习领域。
    你的分析帮助全球顶尖公司做出重要决策。""",
    verbose=True,
    allow_delegation=True,
    memory=True  # ✅ 启用基础记忆
)

# 创建任务
research_task = Task(
    description='研究深度学习领域的最新发展趋势，特别关注大语言模型',
    agent=research_agent,
    expected_output='一份详细的研究报告，包含至少5个关键发现和3个未来预测'
)

# 创建并运行 Crew
crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    memory=True,  # ✅ 启用整体记忆功能
    verbose=2     # 显示更详细的日志
)

result = crew.kickoff()
print(result)
```

#### 方法 B：自定义记忆配置（高级用户）

```python
from crewai import Crew, Agent, Task
from crewai.memory import LongTermMemory, ShortTermMemory, EntityMemory
from crewai.memory.long_term.memory_storage import ChromaMemoryStorage
import os

# 配置长期记忆存储
long_term_memory = LongTermMemory(
    enabled=True,
    storage=ChromaMemoryStorage(collection_name="crew_memory"),
    max_tokens=100000,  # 最大 token 数量
    token_buffer_size=1000  # 缓冲区大小
)

# 配置短期记忆
short_term_memory = ShortTermMemory(
    enabled=True,
    max_tokens=5000
)

# 配置实体记忆
entity_memory = EntityMemory(
    enabled=True,
    storage=ChromaMemoryStorage(collection_name="entity_memory")
)

# 创建带有详细记忆配置的 Agent
deep_learning_expert = Agent(
    role='深度学习专家',
    goal='分析和解释复杂的深度学习概念和技术',
    backstory="""你是DeepLearningAI的首席技术专家Andrew Ng团队的成员，
    拥有深厚的神经网络知识和实践经验。""",
    verbose=True,
    allow_delegation=False,
    long_term_memory=long_term_memory,
    short_term_memory=short_term_memory,
    entity_memory=entity_memory
)

# 定义多个相关任务
task1 = Task(
    description='研究 transformer 架构的工作原理',
    agent=deep_learning_expert,
    expected_output='详细的 transformer 架构说明'
)

task2 = Task(
    description='分析注意力机制在不同应用场景的实现差异',
    agent=deep_learning_expert,
    expected_output='比较分析报告'
)

task3 = Task(
    description='总结研究成果并提出优化建议',
    agent=deep_learning_expert,
    expected_output='综合结论和建议'
)

# 创建 Crew 并启用完整记忆系统
crew_with_memory = Crew(
    agents=[deep_learning_expert],
    tasks=[task1, task2, task3],
    long_term_memory=long_term_memory,
    short_term_memory=short_term_memory,
    entity_memory=entity_memory,
    memory=True,
    verbose=2,
    process='sequential'  # 顺序执行任务以更好地利用记忆
)

# 开始执行
result = crew_with_memory.kickoff()
```

### 第 3 步：检索和利用已存储的记忆

```python
# 检索长期记忆
retrieved_info = long_term_memory.retrieve("transformer architecture key concepts", k=5)

# 检索实体记忆
entity_info = entity_memory.retrieve_entities(["Andrew Ng", "DeepLearningAI"])

# 查看短期记忆摘要
summary = short_term_memory.get_summary()

print("=== 检索到的长期记忆 ===")
for item in retrieved_info:
    print(f"- {item}")

print("\n=== 实体记忆 ===")
print(entity_info)

print("\n=== 短期记忆摘要 ===")
print(summary)
```

---

## 📚 不同类型记忆的详细说明

| 记忆类型 | 用途 | 存储方式 | 适用场景 |
|---------|------|---------|---------|
| **Short-Term Memory** | 当前会话上下文 | RAM + 可选文件 | 多轮对话、连续任务 |
| **Long-Term Memory** | 持久化重要信息 | ChromaDB/向量库 | 跨会话知识积累 |
| **Entity Memory** | 特定实体信息 | ChromaDB | 人物、组织、产品跟踪 |

---

## ⚙️ 环境变量配置

为了让记忆系统正常工作，您可能需要设置以下环境变量：

```bash
# .env 文件或命令行导出
export CHROMA_SERVER_HOST=localhost
export CHROMA_SERVER_PORT=8000
export MEMORY_ENABLED=true
export MAX_MEMORY_TOKENS=100000
```

---

## 🐛 常见问题和解决方案

### 问题 1：记忆没有保存或无法检索

**解决方案：**
- 确保 `memory=True` 参数在 Crew 初始化时正确设置
- 检查 ChromaDB 是否正在运行
- 验证内存存储空间是否足够

```python
# 调试代码
import logging
logging.basicConfig(level=logging.DEBUG)

# 添加记忆状态检查
if crew.long_term_memory.enabled:
    print("✓ 长期记忆已启用")
else:
    print("✗ 长期记忆未启用，请检查配置")
```

### 问题 2：记忆占用过多资源

**解决方案：**
- 调整 `max_tokens` 限制
- 定期清理过期记忆
- 使用分片策略

```python
# 清理旧记忆示例
long_term_memory.delete_older_than(days=30)
```

### 问题 3：在多任务环境下记忆混乱

**解决方案：**
- 为不同任务链使用不同的 collection name
- 使用实体记忆区分不同主题
- 实施记忆命名空间隔离

---

## 🎯 最佳实践建议

### ✅ 应该做的：
1. **明确定义记忆范围** - 只存储真正重要的信息
2. **定期维护记忆库** - 清理过时或不相关的条目
3. **监控内存使用情况** - 防止资源耗尽
4. **测试记忆功能** - 在真实场景下验证效果
5. **使用适当的向量模型** - 根据数据类型选择 embedding 模型

### ❌ 避免的：
1. 不要存储敏感个人信息（PII）到记忆系统
2. 不要无限增长记忆而不加管理
3. 不要在非必要时启用所有三种记忆类型
4. 不要忘记备份重要记忆数据

---

## 📖 参考资料

以下是构建此答案时使用的所有参考资料：

1. **官方文档**: [https://docs.crewai.com](https://docs.crewai.com)
   - Kickoff Crew 页面: [https://docs.crewai.com/en/enterprise/guides/kickoff-crew](https://docs.crewai.com/en/enterprise/guides/kickoff-crew)
   
2. **API 参考**: 
   - Memory API: 提供了 LongTermMemory, ShortTermMemory, EntityMemory 的详细接口
   
3. **GitHub 仓库**: 
   - [crewAIInc/crewAI](https://github.com/crewAIInc/crewAI) - 包含最新的代码实现和示例

4. **ChromaDB 文档**: 
   - 向量存储配置和最佳实践

---

## 🤝 后续支持

Andrew，如果您在实施过程中遇到任何问题，或者需要进一步的定制化建议，请随时联系我！我们的企业支持团队也会为您提供：

- ✨ **24/7 技术支持**
- 📞 **专属客户成功经理**
- 🎓 **定制培训会议**
- 📊 **性能优化咨询**

您可以通过以下方式联系我们：
- 📧 support@crewai.com
- 🚀 Slack Enterprise Channel
- 🏢 预约演示：[https://app.crewai.com/demo](https://app.crewai.com/demo)

再次感谢您的信任，我们期待与 DeepLearningAI 建立长期合作关系！如果您还有其他问题，欢迎随时提出。

祝您项目顺利推进！

**Warmest regards,**

您的 CrewAI 支持代表  
🌟 致力于为 DeepLearningAI 提供最优质的支持服务